# Margit Fight — Trajectory EDA

Load any observer JSON artifact from `logs/observer/` and produce a full EDA with improvement suggestions.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (14, 4)

# ── CONFIG ──────────────────────────────────────────────────────────────────
# Leave ARTIFACT_PATH as None to auto-pick the latest artifact
ARTIFACT_PATH = None

# Animation IDs that count as 'attacking'
ATTACK_PREFIXES = ('290',)
# Animation IDs that count as 'dodging'
DODGE_IDS = {'27100', '27101', '27102', '27200'}
# Animation IDs that count as 'guarding'
GUARD_IDS = {'21000', '21001', '21002'}
# ────────────────────────────────────────────────────────────────────────────

def _latest_json() -> Path:
    root = Path('..') if Path('../logs/observer').exists() else Path('.')
    candidates = sorted(root.glob('logs/observer/*.json'))
    return candidates[-1] if candidates else None

artifact_path = Path(ARTIFACT_PATH) if ARTIFACT_PATH else _latest_json()
print(f'Loading: {artifact_path}')

payload = json.loads(artifact_path.read_text(encoding='utf-8'))
meta = payload.get('summary', {})
df = pd.DataFrame(payload['records'])

# Explode pose columns
if 'player_pose' in df.columns and df['player_pose'].iloc[0] is not None:
    df[['px', 'py', 'pz', 'pa']] = pd.DataFrame(df['player_pose'].tolist(), index=df.index)
if 'boss_pose' in df.columns and df['boss_pose'].iloc[0] is not None:
    df[['bx', 'by', 'bz', 'ba']] = pd.DataFrame(df['boss_pose'].tolist(), index=df.index)

if 'px' in df.columns and 'bx' in df.columns:
    df['dist_to_boss'] = np.sqrt((df['px'] - df['bx'])**2 + (df['py'] - df['by'])**2)

df['player_hp_delta'] = df['player_hp'].diff().fillna(0)
df['boss_hp_delta'] = df['boss_hp'].diff().fillna(0)

print(f"Steps: {len(df)}  |  Status: {meta.get('status')}  |  Runtime: {meta.get('runtime')}")
df.head(3)

## 1 — HP & Stamina Timelines

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

# Player HP
hit_steps = df[df['player_hp_delta'] < 0]['step']
axes[0].plot(df['step'], df['player_hp'], color='steelblue', label='Player HP')
axes[0].vlines(hit_steps, 0, df['player_max_hp'].iloc[0], color='red', alpha=0.3, linewidth=1, label='Hit taken')
axes[0].set_ylabel('Player HP')
axes[0].legend(loc='upper right')
axes[0].set_title('Player HP over time')

# Boss HP
dmg_steps = df[df['boss_hp_delta'] < 0]['step']
axes[1].plot(df['step'], df['boss_hp'], color='firebrick', label='Boss HP')
axes[1].vlines(dmg_steps, df['boss_hp'].min(), df['boss_max_hp'].iloc[0], color='orange', alpha=0.3, linewidth=1, label='Damage dealt')
axes[1].set_ylabel('Boss HP')
axes[1].legend(loc='upper right')
axes[1].set_title('Boss HP over time')

# Stamina
axes[2].plot(df['step'], df['player_sp'], color='goldenrod', label='Stamina')
axes[2].set_ylabel('Stamina')
axes[2].set_xlabel('Step')
axes[2].legend(loc='upper right')
axes[2].set_title('Stamina over time')

plt.tight_layout()
plt.show()

## 2 — Positioning

In [ ]:
if 'px' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # XY trajectory
    sc = axes[0].scatter(df['px'], df['py'], c=df['step'], cmap='viridis', s=10, alpha=0.7)
    axes[0].scatter(df['bx'], df['by'], c=df['step'], cmap='Reds', s=10, alpha=0.4, marker='x')
    plt.colorbar(sc, ax=axes[0], label='Step')
    axes[0].set_title('XY Trajectory (blue=player, red=boss)')
    axes[0].set_xlabel('X')
    axes[0].set_ylabel('Y')

    # Distance to boss over time
    axes[1].plot(df['step'], df['dist_to_boss'], color='purple')
    axes[1].axhline(df['dist_to_boss'].mean(), color='purple', linestyle='--', alpha=0.5, label=f"mean={df['dist_to_boss'].mean():.1f}")
    axes[1].set_title('Distance to Boss over time')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Distance')
    axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print('No pose data available in this artifact.')

## 3 — Animation Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Player animations
pa = Counter(df['player_animation'].astype(str))
top_pa = pd.Series(dict(pa.most_common(15)))
top_pa.sort_values().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 15 Player Animations')
axes[0].set_xlabel('Count')

# Boss animations
ba = Counter(df['boss_animation'].astype(str))
top_ba = pd.Series(dict(ba.most_common(15)))
top_ba.sort_values().plot(kind='barh', ax=axes[1], color='firebrick')
axes[1].set_title('Top 15 Boss Animations')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

## 4 — Event Analysis

In [ ]:
hits = df[df['player_hp_delta'] < 0][['step', 'player_hp_delta', 'player_animation', 'boss_animation', 'dist_to_boss'] if 'dist_to_boss' in df.columns else ['step', 'player_hp_delta', 'player_animation', 'boss_animation']].copy()
hits['damage_received'] = -hits['player_hp_delta']
hits = hits.drop(columns='player_hp_delta')

damage_dealt = df[df['boss_hp_delta'] < 0][['step', 'boss_hp_delta', 'player_animation']].copy()
damage_dealt['damage_dealt'] = -damage_dealt['boss_hp_delta']
damage_dealt = damage_dealt.drop(columns='boss_hp_delta')

print(f"=== Hit events ({len(hits)} total, {hits['damage_received'].sum():.0f} total dmg received) ===")
print(hits.to_string(index=False))
print()
print(f"=== Damage dealt events ({len(damage_dealt)} total, {damage_dealt['damage_dealt'].sum():.0f} total dmg dealt) ===")
print(damage_dealt.to_string(index=False))

## 5 — Improvement Suggestions

In [ ]:
suggestions = []
sep = '-' * 60

# ── Stamina management ──────────────────────────────────────────
sp_min = df['player_sp'].min()
sp_max = df['player_max_sp'].iloc[0]
sp_depletion_steps = (df['player_sp'] < sp_max * 0.2).sum()
if sp_depletion_steps > 5:
    suggestions.append(
        f'STAMINA: Stamina dropped below 20% for {sp_depletion_steps} steps '
        f'(min={sp_min:.0f}/{sp_max:.0f}). '
        'Consider spacing out attacks and dodges to allow stamina recovery.'
    )
else:
    suggestions.append(f'STAMINA: Good stamina management (min={sp_min:.0f}/{sp_max:.0f}, low-steps={sp_depletion_steps}).')

# ── Aggression ratio ─────────────────────────────────────────────
attack_steps = df['player_animation'].astype(str).str.startswith(ATTACK_PREFIXES).sum()
dodge_steps = df['player_animation'].astype(str).isin(DODGE_IDS).sum()
total = len(df)
atk_ratio = attack_steps / total
dodge_ratio = dodge_steps / total
suggestions.append(
    f'AGGRESSION: Attacking {atk_ratio:.1%} of steps, dodging {dodge_ratio:.1%} of steps. '
    + ('Try to increase attack frequency when boss is recovering.' if atk_ratio < 0.15 else 'Reasonable aggression level.')
)

# ── Hit exposure ─────────────────────────────────────────────────
if len(hits) > 0:
    avg_dmg = hits['damage_received'].mean()
    suggestions.append(
        f'HIT EXPOSURE: Took {len(hits)} hits averaging {avg_dmg:.0f} dmg each. '
        + ('High damage per hit — boss attacks may be hitting during recovery frames. Consider rolling earlier.' if avg_dmg > 120 else 'Damage per hit is moderate.')
    )
else:
    suggestions.append('HIT EXPOSURE: No hits taken — excellent defensive play.')

# ── DPS efficiency ───────────────────────────────────────────────
total_dealt = -df['boss_hp_delta'][df['boss_hp_delta'] < 0].sum()
runtime_s = len(df) * 0.2  # interval
dps = total_dealt / runtime_s if runtime_s > 0 else 0
suggestions.append(
    f'DPS: Dealt {total_dealt:.0f} total boss damage in {runtime_s:.0f}s ({dps:.1f} dmg/s). '
    + ('Low DPS — look for more opening windows after dodging boss attacks.' if dps < 5 else 'Good damage output.')
)

# ── Positioning ──────────────────────────────────────────────────
if 'dist_to_boss' in df.columns:
    avg_dist = df['dist_to_boss'].mean()
    suggestions.append(
        f'POSITIONING: Average distance to boss = {avg_dist:.1f} units. '
        + ('Playing at long range — staying closer allows more attack opportunities.' if avg_dist > 8 else 'Good proximity to boss.')
    )

# ── Print ─────────────────────────────────────────────────────────
print(sep)
print('FIGHT ANALYSIS — IMPROVEMENT SUGGESTIONS')
print(sep)
for i, s in enumerate(suggestions, 1):
    print(f'{i}. {s}')
    print()
print(sep)